# Response Streaming

Streaming responses solve the problem where users may experience long wait times to get a full response from a chatbot. This is done by generating the text in chunks.

With this approach, Claude will send an immediate response to generate text, then will send series of events leading to the completed, overall response. These are all done in one API request to Claude.

## Understanding Stream Events 

There are several event types that Claude can send:
- **MessageStart** - a new message being sent
- **ContentBlockStart** - start of new block containing text, tool use or other content
- **ContentBlockDelta** - The chunks of generated text from Claude
- **ContentBlockStop** - Current content block completed
- **MessageDelta** - Current message has been complete
- **MessageStop** - End of information on current message

The **ContentBlockDelta** events are the generated text to send to users

## Streaming Implementation

In [4]:
from utils.util_funcs import *

client, model = api_client_setup()

Basic implementation

In [5]:
messages = []

add_user_message(messages, "Write a 1 sentence description of a fake database")

stream = client.messages.create(
    model = model,
    max_tokens = 1000, 
    messages = messages,
    stream = True
)

for event in stream:
    print(event)

RawMessageStartEvent(message=Message(id='msg_011Cd4kuKriREaPfwEFEv15N', container=None, content=[], model='claude-sonnet-5', role='assistant', stop_details=None, stop_reason=None, stop_sequence=None, type='message', usage=Usage(cache_creation=CacheCreation(ephemeral_1h_input_tokens=0, ephemeral_5m_input_tokens=0), cache_creation_input_tokens=0, cache_read_input_tokens=0, inference_geo='global', input_tokens=21, output_tokens=1, output_tokens_details=None, server_tool_use=None, service_tier='standard')), type='message_start')
RawContentBlockStartEvent(content_block=TextBlock(citations=None, text='', type='text'), index=0, type='content_block_start')
RawContentBlockDeltaEvent(delta=TextDelta(text='A', type='text_delta'), index=0, type='content_block_delta')
RawContentBlockDeltaEvent(delta=TextDelta(text=' lightweight, cloud-native database called "NimbusDB" that promises mill', type='text_delta'), index=0, type='content_block_delta')
RawContentBlockDeltaEvent(delta=TextDelta(text='isecon

Simplified approach with the streaming interface

In [6]:
with client.messages.stream(
    model = model,
    max_tokens = 1000, 
    messages = messages
) as stream:
    for text in stream.text_stream:
        print(text, end = "")

A lightweight, self-organizing database called "EchoStore" that indexes data based on how frequently it's accessed, automatically pushing rarely-used records into a compressed archive layer while keeping hot data instantly available in memory.

Getting the full message content from the stream

In [7]:
final_message = stream.get_final_message()
final_message.content[0].text

'A lightweight, self-organizing database called "EchoStore" that indexes data based on how frequently it\'s accessed, automatically pushing rarely-used records into a compressed archive layer while keeping hot data instantly available in memory.'